In [2]:
# pipeline base params
table_name = "title_data_silver"

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import functions as F
import re

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 5, Finished, Available, Finished)

In [4]:
df = spark.read.format("csv").option("header","true").load("abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/TitleMaster.csv")
# df now is a Spark DataFrame containing CSV data from "abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/TitleMaster.csv".
display(df)

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bf78c66e-5787-4b28-9a86-2c633e28a83c)

In [5]:
# Silver Transformation
# Standardize columns
def standardize_columns(df):
    for c in df.columns:
        new_c = c.lower().strip().replace(" ", "_")
        new_c = re.sub(r'[^a-z0-9_]', '', new_c)
        df = df.withColumnRenamed(c, new_c)
    return df

title_df = standardize_columns(df)

# Remove leading * from title description
title_df = title_df.withColumn(
    "titledescription",
    F.regexp_replace(F.col("titledescription"), r'^\*', '')
)

# Remove duplicates & nulls
title_df = title_df.dropDuplicates(["titlecode"]) \
                   .filter(F.col("titlecode").isNotNull())

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 7, Finished, Available, Finished)

In [6]:
# Write Silver
title_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("Silver Title completed")

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 8, Finished, Available, Finished)

Silver Title completed


In [7]:
display(title_df)

StatementMeta(, cad53b5d-d725-441b-a2fa-33af53a904f2, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, df04fac3-55a0-4727-981d-834b365e6890)